# OAI XRAY TXT Schema Explorer
Thin notebook wrapper around `tmc_oai` schema APIs.


In [7]:
from pathlib import Path
import re

import pandas as pd
from IPython.display import HTML, Markdown, display

from tmc_oai import (
    build_hover_table_html,
    build_schema_comparison,
    build_schema_explorer,
    load_oai_env,
    load_package_timepoint_map,
    resolve_package_selection,
)


In [8]:
TIMEPOINT_MAP_CSV = None  # Optional override path; default comes from .oai_env.json
PACKAGE_NUMBER = ""
TIMEPOINT_LABEL = "XRAYMETA"
FOCUS_FILE = "oai_kxrsemiquant01.txt"
COMPARE_FILES = ["oai_kxrsemiquant01.txt", "oai_hxrsemiquant01.txt"]
FILE_REGEX = ""  # Optional regex for summary filtering


In [9]:
cfg = load_oai_env()
map_csv = Path(TIMEPOINT_MAP_CSV).expanduser().resolve() if TIMEPOINT_MAP_CSV else cfg.timepoint_map_csv

package_map = load_package_timepoint_map(map_csv)
resolved_package_number, resolved_timepoint_label = resolve_package_selection(
    package_map,
    package_number=PACKAGE_NUMBER,
    timepoint_label=TIMEPOINT_LABEL,
)

package_dir = cfg.oai_dataset_root / f"Package_{resolved_package_number}"
schema_result = build_schema_explorer(package_dir)
comparison = build_schema_comparison(schema_result, COMPARE_FILES[0], COMPARE_FILES[1])

print(f"Dataset root: {cfg.oai_dataset_root}")
print(f"Map CSV: {map_csv}")
print(f"Resolved package number: {resolved_package_number}")
print(f"Resolved timepoint label: {resolved_timepoint_label}")
print(f"Package dir: {package_dir}")

summary_df = schema_result.summary_df.copy()
if FOCUS_FILE.strip():
    summary_df["_focus"] = summary_df["file_name"].astype(str).str.lower().ne(FOCUS_FILE.strip().lower())
    summary_df = summary_df.sort_values(["_focus", "file_name"], key=lambda s: s.astype(str).str.lower())
    summary_df = summary_df.drop(columns=["_focus"])
else:
    summary_df = summary_df.sort_values("file_name", key=lambda s: s.astype(str).str.lower())

if FILE_REGEX.strip():
    pattern = re.compile(FILE_REGEX.strip(), flags=re.IGNORECASE)
    summary_df = summary_df.loc[
        summary_df["file_name"].astype(str).map(lambda value: bool(pattern.search(value)))
    ].reset_index(drop=True)

display(Markdown("## Package File Summary"))
display(summary_df.reset_index(drop=True))


Dataset root: X:\OAI
Map CSV: D:\TMCProject_V2\OAI\reference\oai_package_timepoint_map.csv
Resolved package number: 1243845
Resolved timepoint label: XRAYMETA
Package dir: X:\OAI\Package_1243845


## Package File Summary

,file_name,parse_type,delimiter,column_count,description_row_detected,preview_row_count
0,oai_kxrsemiquant01.txt,tabular,\t,34,True,5
1,dataset_collection.txt,tabular,\t,5,True,5
2,oai_enrollee01.txt,tabular,\t,17,True,5
3,oai_hxrsemiquant01.txt,tabular,\t,27,True,5
4,oai_inventory01.txt,tabular,\t,82,True,5
5,oai_kxrquant_ftb01.txt,tabular,\t,21,True,5
6,oai_kxrquantjsw01.txt,tabular,\t,46,True,5
7,oai_outcome01.txt,tabular,\t,106,True,5
8,oai_xralign01.txt,tabular,\t,21,True,5
9,oai_xrmeta01.txt,tabular,\t,25,True,5


In [10]:
display(Markdown(f"## Summary of `{comparison.left_file}` and `{comparison.right_file}`"))

display(Markdown("### Shared Columns"))
display(comparison.shared_summary_df)

display(Markdown(f"### Unique Columns in `{comparison.left_file}`"))
if comparison.left_unique_summary_df.empty:
    print("No unique columns.")
else:
    display(comparison.left_unique_summary_df)

display(Markdown(f"### Unique Columns in `{comparison.right_file}`"))
if comparison.right_unique_summary_df.empty:
    print("No unique columns.")
else:
    display(comparison.right_unique_summary_df)


## Summary of `oai_kxrsemiquant01.txt` and `oai_hxrsemiquant01.txt`

### Shared Columns

,column_name,description,value_type,oai_kxrsemiquant01.txt summary,oai_hxrsemiquant01.txt summary
0,ageyears,Age in years,numeric,"range=[45, 87] | mean=63.182 | unique=43 | mis...","range=[45, 84] | mean=62.83 | unique=40 | miss..."
1,barcode,Barcode of image analyzed,numeric,"range=[16600000104, 16604340202] | mean=166021...","range=[16600002705, 16603750103] | mean=166017..."
2,collection_id,collection_id,numeric,"range=[2343, 2343] | mean=2343 | unique=1 | mi...","range=[2343, 2343] | mean=2343 | unique=1 | mi..."
3,collection_title,collection_title,categorical,values=[Osteoarthritis Initiative] | unique=1 ...,values=[Osteoarthritis Initiative] | unique=1 ...
4,dataset_id,dataset_id,numeric,"range=[55677, 55677] | mean=55677 | unique=1 |...","range=[56486, 56486] | mean=56486 | unique=1 |..."
5,interview_age,Age in months at the time of the interview/tes...,numeric,"range=[540, 1044] | mean=758.187 | unique=43 |...","range=[540, 1008] | mean=753.964 | unique=40 |..."
6,interview_date,Date on which the interview/genetic test/sampl...,categorical,"values=[01/02/2007, 01/02/2008, 01/02/2009, 01...","values=[01/02/2009, 01/03/2005, 01/03/2006, 01..."
7,readprj,Project,numeric,"range=[15, 42] | mean=21.201 | unique=3 | miss...","range=[21, 21] | mean=21 | unique=1 | missing=0"
8,sex,Sex of subject at birth,categorical,"values=[F, M] | unique=2 | missing=0","values=[F, M] | unique=2 | missing=0"
9,side,Side,numeric,"range=[1, 2] | mean=1.502 | unique=2 | missing=0","range=[1, 2] | mean=1.5 | unique=2 | missing=0"


### Unique Columns in `oai_kxrsemiquant01.txt`

,column_name,description,value_type,value_summary
0,oai_kxrsemiquant01_id,oai_kxrsemiquant01_id,numeric,"range=[1, 57343] | mean=28672 | unique=57343 |..."
1,xrattl,Attrition (OARSI grades 0- 3) tibia lateral co...,numeric,"range=[0, 3] | mean=0.006 | unique=4 | missing..."
2,xrattm,Attrition (OARSI grades 0- 3) tibia medial com...,numeric,"range=[0, 3] | mean=0.012 | unique=4 | missing..."
3,xrchl,Chondrocalcinosis (Grades 0- 1) lateral compar...,numeric,"range=[0, 1] | mean=0.055 | unique=2 | missing..."
4,xrchm,Chondrocalcinosis (Grades 0- 1) medial compart...,numeric,"range=[0, 1] | mean=0.053 | unique=2 | missing..."
5,xrcyfl,Cysts (Grades 0- 1) femur lateral compartment,numeric,"range=[0, 1] | mean=0.005 | unique=2 | missing..."
6,xrcyfm,Cysts (Grades 0- 1) femur medial compartment,numeric,"range=[0, 1] | mean=0.017 | unique=2 | missing..."
7,xrcytl,Cysts (Grades 0- 1) tibia lateral compartment,numeric,"range=[0, 1] | mean=0.012 | unique=2 | missing..."
8,xrcytm,Cysts (Grades 0- 1) tibia medial compartment,numeric,"range=[0, 2] | mean=0.066 | unique=3 | missing..."
9,xrjsl,Joint space narrowing (OARSI grades 0- 3) late...,numeric,"range=[0, 3] | mean=0.127 | unique=13 | missing=3"


### Unique Columns in `oai_hxrsemiquant01.txt`

,column_name,description,value_type,value_summary
0,oai_hxrsemiquant01_id,oai_hxrsemiquant01_id,numeric,"range=[1, 16760] | mean=8380.5 | unique=16760 ..."
1,xrhaosi,Hip Inferior acetabular osteophytes,numeric,"range=[0, 3.03] | mean=0.063 | unique=4 | miss..."
2,xrhaoss,Hip Superior acetabular osteophytes,numeric,"range=[0, 3.03] | mean=0.316 | unique=5 | miss..."
3,xrhcya,Hip Acetabular subchondral cysts,numeric,"range=[0, 1] | mean=0.019 | unique=2 | missing=0"
4,xrhcyf,Hip Femoral subchondral cysts,numeric,"range=[0, 1] | mean=0.008 | unique=2 | missing=0"
5,xrhflat,Hip Femoral head flattening/deformity,numeric,"range=[0, 1] | mean=0.002 | unique=2 | missing=0"
6,xrhfosi,Hip Inferior femoral osteophytes,numeric,"range=[0, 3.03] | mean=0.041 | unique=6 | miss..."
7,xrhfoss,Hip Superior femoral osteophytes,numeric,"range=[0, 3.03] | mean=0.14 | unique=5 | missi..."
8,xrhip_status,Hip radiographic status,numeric,"range=[0, 2] | mean=0.248 | unique=3 | missing=0"
9,xrhjsnsl,Hip Supero-lateral joint space narrowing (JSN),numeric,"range=[0, 3.5] | mean=0.111 | unique=8 | missi..."
